# NDgpu — HP-MR 3D: S_N vs diffusion drum-worth convergence (Colab)

Does S_N transport agree with diffusion on the HP-MR control-drum worth, and
where do the transport effects show? On a **coarse** mesh the S_N-vs-diffusion
drum worth is dominated by discretisation (step over-leakage), and S_N even
*over*-predicts the worth -- an artifact. As the prism mesh refines, the
schemes converge and the genuine **transport self-shielding** emerges: the
strong B4C drum arc depresses the flux inside itself, so S_N resolves *less*
worth than diffusion (diffusion can't see the self-shielding).

This study sweeps the in-plane refinement and reports, for diffusion and for
S_N (step = 1st order, SCB = 2nd order), the **drum worth** (reactivity of
drums withdrawn vs inserted) and the **correction** `S_N - diffusion`. The
signature to watch: the correction starting positive (coarse-mesh artifact) and
crossing **negative** (physical self-shielding) as `refine` grows -- with SCB
reaching it sooner than step.

Runs the full GPU stack: level-scheduled prism sweeps + CUDA graphs, device
DSA, device multigrid CMFD (`cmfd_solver="mg"`, the O(N) solve for the large 3D
drift matrices). Cross sections are illustrative placeholders; drum WORTH is the
right quantity to compare (systematic k offsets cancel).

In [ ]:
import os
try:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    get_ipython().run_line_magic("pip", f"install -q {zip_name}")
    try:
        import cupy
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q cupy-cuda12x")
    get_ipython().run_line_magic("pip", "install -q pyamg")
    get_ipython().system("nvidia-smi -L")
except ImportError:
    pass

In [ ]:
import time
import numpy as np
from ndgpu.benchmarks import build_hpmr3d
from ndgpu.tri import TriDiffusionEigenSolver
from ndgpu.tri_sn import TriSNTransportSolver

try:
    import cupy
    HAVE_GPU = cupy.cuda.runtime.getDeviceCount() > 0
except Exception:
    HAVE_GPU = False
DEV = "gpu" if HAVE_GPU else "cpu"
print("device:", DEV)

QUICK = bool(os.environ.get("NDGPU_QUICK"))
# in-plane refinement sweep -- refine>=4 is where the transport self-shielding
# becomes meaningful (coarse refine=2 is artifact-dominated). SCB is ~3x the
# DoF of step; a 16 GB GPU handles refine 4/6/8 at nz=10.
REFINES = [2, 4] if QUICK else [4, 6, 8]
NZ = 10
SCHEMES = ["step", "scb"]
QUAD = dict(n_polar=2, n_azi=8)               # 16 ordinates
TOL = dict(tol_k=5e-6, tol_source=5e-5, max_outer=120)


def build(refine, angle):
    return build_hpmr3d(refine=refine, nz=NZ, drum_angle_deg=angle,
                        absorber="polar")     # 0 deg = drums inserted, 180 = withdrawn


def diff_k(p):
    return TriDiffusionEigenSolver(
        p.grid, p.materials, p.material_map, active=p.active, mask_bc=p.mask_bc,
        bc=p.bc, mix_material=p.mix_material, mix_weight=p.mix_weight,
        device=DEV).solve(tol_k=1e-6, tol_source=1e-5).k_eff


def sn_k(p, scheme):
    # bc="vacuum" + active mask == the physical boundary (excised void border +
    # vacuum z faces); levels engine + device MG CMFD on the GPU.
    s = TriSNTransportSolver(
        p.grid, p.materials, p.material_map, active=p.active, bc="vacuum",
        mix_material=p.mix_material, mix_weight=p.mix_weight, scheme=scheme,
        engine="levels", device=DEV, cmfd_solver="mg", **QUAD)
    r = s.solve(**TOL)
    assert r.converged, f"S_N {scheme} refine={p.grid.shape} not converged"
    return r.k_eff


def worth(k_in, k_out):                        # pcm: withdrawn minus inserted
    return (1.0 / k_in - 1.0 / k_out) * 1e5


if HAVE_GPU:                                    # warm-up (kernels, graphs, pools)
    p = build(REFINES[0], 0.0); sn_k(p, "step"); sn_k(p, "scb")

In [ ]:
rows = []
for refine in REFINES:
    t0 = time.perf_counter()
    p_in, p_out = build(refine, 0.0), build(refine, 180.0)
    cells = int(p_in.active.sum())
    dw = worth(diff_k(p_in), diff_k(p_out))
    row = dict(refine=refine, cells=cells, diff=dw)
    for sch in SCHEMES:
        w = worth(sn_k(p_in, sch), sn_k(p_out, sch))
        row[f"sn_{sch}"] = w
        row[f"corr_{sch}"] = w - dw
    row["t"] = time.perf_counter() - t0
    rows.append(row)
    print(f"refine={refine} ({cells} prisms) done in {row['t']:.0f}s")

print(f"\n{'refine':>6} {'prisms':>7} {'diff':>7} {'SN_step':>8} {'SN_scb':>8} "
      f"{'step-diff':>9} {'scb-diff':>9}")
for r in rows:
    print(f"{r['refine']:>6} {r['cells']:>7} {r['diff']:>7.0f} {r['sn_step']:>8.0f} "
          f"{r['sn_scb']:>8.0f} {r['corr_step']:>+9.0f} {r['corr_scb']:>+9.0f}")
print("\n(worths in pcm; corr = S_N - diffusion. corr>0 = coarse-mesh artifact "
      "over-predicting worth; corr<0 = physical transport self-shielding.)")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
x = [r["refine"] for r in rows]
ax[0].plot(x, [r["diff"] for r in rows], "o-", label="diffusion")
ax[0].plot(x, [r["sn_step"] for r in rows], "s-", label="S_N step")
ax[0].plot(x, [r["sn_scb"] for r in rows], "^-", label="S_N SCB")
ax[0].set(xlabel="in-plane refine", ylabel="drum worth [pcm]",
          title="HP-MR drum worth: S_N vs diffusion")
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].axhline(0.0, color="k", lw=0.8, ls="--")
ax[1].plot(x, [r["corr_step"] for r in rows], "s-", label="step - diffusion")
ax[1].plot(x, [r["corr_scb"] for r in rows], "^-", label="SCB - diffusion")
ax[1].set(xlabel="in-plane refine", ylabel="S_N - diffusion worth [pcm]",
          title="transport correction (crosses <0 = self-shielding)")
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.show()

## Reading the results

* **Drum worth converges**: as `refine` grows, S_N (step and SCB) and diffusion
  agree on the broad worth; the coarse-mesh spread shrinks. SCB (2nd order)
  tracks the converged value from a coarser mesh than step.
* **The transport correction `S_N - diffusion`** is the physics. It starts
  **positive** on coarse meshes (step over-leakage makes S_N over-predict the
  worth -- an artifact) and should cross **negative** as the mesh sharpens:
  genuine **self-shielding** of the B4C drum arc, which diffusion cannot see.
  SCB reaches the physical (negative) regime at a lower `refine` than step.
* **Absolute k** is *not* the comparison to trust -- it carries a systematic
  step-leakage offset that cancels in the worth. That is why this study reports
  worth, per the benchmark-reporting rule.

If the SCB correction is still positive at the largest `refine` here, push
`REFINES` higher (memory permitting) -- the crossover is the meaningful result,
and it is what a GPU run buys you over the memory-limited CPU.